# H4 Ablation: Which Component Breaks the Distribution-Aware Framework?

This notebook answers that question directly with an ablation: four coordinate-aware variants of self-training, isolating each component,

| Variant | Re-weighting | Spatial weighting | Adaptive threshold |
|---|---|---|---|
| `reweighted_self_training` | ✓ | | |
| `reweighted_spatial_self_training` | ✓ | ✓ | |
| `reweighted_adaptive_self_training` | ✓ | | ✓ |
| `full_distribution_aware` | ✓ | ✓ | ✓ |

run alongside plain `self_training` across all five settings used throughout the manuscript (synthetic pilot, PovertyMap-WILDS, housing, socio-economic, air quality), at **20 seeds per cell** rather than the original 3 -- the statistical-rigor fix the review also requested, so every accuracy-drop number below carries an honest seed-cluster bootstrap confidence interval (`ssl_spatial.metrics.bootstrap`) rather than being a bare point estimate.

New code for this pass: `ssl_spatial.models.distribution_aware.fit_reweighted_spatial_self_training` and `fit_reweighted_adaptive_self_training` (additive -- the original `fit_reweighted_self_training` and `fit_full_distribution_aware` are untouched), `configs/h4_ablation_v2.yaml`, and `ssl_spatial.experiments.h4_ablation_v2` (a thin wrapper reusing `spatial_methods_comparison.py`'s sweep machinery unchanged).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd

from ssl_spatial.metrics.bootstrap import bootstrap_ci_drop
from ssl_spatial.plotting import (
    BASELINE_COLOR, FONT_LEGEND, FONT_TICK, FONT_LABEL, INK_SECONDARY,
    METHOD_COLORS, METHOD_LABELS, new_figure, savefig, style_axes,
)

FIG_DIR = REPO_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

ABLATION_ORDER = ["self_training", "reweighted_self_training", "reweighted_spatial_self_training",
                   "reweighted_adaptive_self_training", "full_distribution_aware"]
DATASET_ORDER = ["synthetic", "wilds", "housing", "socioeconomic", "air_quality"]
DATASET_LABELS = {"synthetic": "Synthetic", "wilds": "PovertyMap-WILDS", "housing": "Housing",
                   "socioeconomic": "Socio-economic", "air_quality": "Air quality"}

print("Repo root:", REPO_ROOT)

## 1. Load the ablation sweep

`configs/h4_ablation_v2.yaml`: 6 mismatch-severity levels × 20 seeds × 5 methods × 5 datasets = 3,000 runs, produced by `python -m ssl_spatial.experiments.h4_ablation_v2 configs/h4_ablation_v2.yaml` (reuses `spatial_methods_comparison.py`'s sweep executor and the unified `method_registry` interface unchanged -- only the config and the two new model functions are new).

In [2]:
df = pd.read_csv(REPO_ROOT / "results" / "h4_ablation_v2.csv")
print(f"{len(df)} rows: {df['dataset'].nunique()} datasets × {df['mismatch_alpha'].nunique()} alphas × "
      f"{df['seed'].nunique()} seeds × {df['method'].nunique()} methods")
df.groupby(["dataset", "method"]).size().unstack()

3000 rows: 5 datasets × 6 alphas × 20 seeds × 5 methods


method,full_distribution_aware,reweighted_adaptive_self_training,reweighted_self_training,reweighted_spatial_self_training,self_training
dataset,,,,,
air_quality,120,120,120,120,120
housing,120,120,120,120,120
socioeconomic,120,120,120,120,120
synthetic,120,120,120,120,120
wilds,120,120,120,120,120


## 2. Accuracy drop per method, per dataset, with bootstrap confidence intervals

For each (dataset, method), the out-of-region accuracy drop from minimal ($\alpha=0$) to maximal ($\alpha=1$) mismatch severity, with a 95% seed-cluster percentile bootstrap CI (`bootstrap_ci_drop`, 200 resamples of the 20 seeds) -- the same resampling scheme already validated against `changepoint_analysis.py`'s existing breakpoint CIs.

In [3]:
drop = bootstrap_ci_drop(df, group_cols=["dataset", "method"], value_col="accuracy_out_region",
                          alpha_col="mismatch_alpha", lo=0.0, hi=1.0, n_boot=200, seed=0)
drop["drop_pp"] = drop["drop"] * 100
drop["ci_low_pp"] = drop["ci_low"] * 100
drop["ci_high_pp"] = drop["ci_high"] * 100
drop["method"] = pd.Categorical(drop["method"], categories=ABLATION_ORDER, ordered=True)
drop["dataset"] = pd.Categorical(drop["dataset"], categories=DATASET_ORDER, ordered=True)
drop = drop.sort_values(["dataset", "method"])

table = drop.copy()
table["method"] = table["method"].map(METHOD_LABELS)
table["dataset"] = table["dataset"].map(DATASET_LABELS)
table["drop (95% CI)"] = table.apply(
    lambda r: f"{r['drop_pp']:.2f} [{r['ci_low_pp']:.2f}, {r['ci_high_pp']:.2f}]", axis=1)
table[["dataset", "method", "drop (95% CI)"]].reset_index(drop=True)

,dataset,method,drop (95% CI)
0,Synthetic,Self-training,"2.42 [0.43, 4.82]"
1,Synthetic,Reweighted self-training,"3.33 [1.18, 6.17]"
2,Synthetic,+ spatial weighting,"7.58 [3.75, 11.23]"
3,Synthetic,+ adaptive threshold,"3.83 [1.70, 6.35]"
4,Synthetic,Full framework (all three),"8.92 [4.66, 13.19]"
5,PovertyMap-WILDS,Self-training,"14.57 [9.72, 21.40]"
6,PovertyMap-WILDS,Reweighted self-training,"12.00 [5.70, 18.74]"
7,PovertyMap-WILDS,+ spatial weighting,"21.67 [16.42, 26.87]"
8,PovertyMap-WILDS,+ adaptive threshold,"12.93 [6.46, 20.67]"
9,PovertyMap-WILDS,Full framework (all three),"21.30 [15.89, 27.17]"


## 3. Isolating the culprit component

If both added components contributed roughly equally to the full framework's regression, the two single-component variants should each show a drop roughly midway between re-weighting-only and the full framework. That is not what the data show.

In [ ]:
fig, ax = new_figure(figsize=(10.5, 5.8))
x = np.arange(len(DATASET_ORDER))
width = 0.16
for i, method in enumerate(ABLATION_ORDER):
    sub = drop[drop["method"] == method].set_index("dataset").loc[DATASET_ORDER]
    offset = (i - (len(ABLATION_ORDER) - 1) / 2) * width
    ax.bar(x + offset, sub["drop_pp"], width, color=METHOD_COLORS[method], label=METHOD_LABELS[method],
           yerr=[sub["drop_pp"] - sub["ci_low_pp"], sub["ci_high_pp"] - sub["drop_pp"]],
           capsize=2, error_kw={"linewidth": 0.8, "ecolor": INK_SECONDARY})
ax.axhline(0, color=BASELINE_COLOR, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([DATASET_LABELS[d] for d in DATASET_ORDER], fontsize=FONT_TICK)
ax.set_ylabel("Accuracy drop, alpha=0 to alpha=1 (pp)", fontsize=FONT_LABEL, color=INK_SECONDARY)
ax.legend(frameon=False, fontsize=FONT_LEGEND - 1, labelcolor=INK_SECONDARY, ncol=2, loc="upper left")
savefig(fig, FIG_DIR / "fig16_h4_ablation_v2.png")
